# Task 1: Exploratory Data Analysis and Data Preprocessing

This notebook explores the full CFPB complaint dataset and produces the cleaned,
filtered output required for the RAG pipeline.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append(os.path.abspath(".."))

from src.eda import narrative_availability, raw_product_distribution, word_count_summary
from src.preprocess import run_pipeline

sns.set_theme(style="whitegrid")
%matplotlib inline

RAW_PATH = "../data/raw/complaints.csv"
FILTERED_PATH = "../data/filtered_complaints.csv"
PROCESSED_PATH = "../data/processed/complaints_clean.csv"
FIG_DIR = "../data/processed"
os.makedirs(FIG_DIR, exist_ok=True)

print(f"Raw CSV size: {os.path.getsize(RAW_PATH) / 1024**3:.2f} GB")

## 1. Exploratory analysis on the raw dataset

Chunked scans avoid loading the full 5.6 GB file into memory.

In [ ]:
narrative_stats = narrative_availability(RAW_PATH)
print(f"Total records:          {narrative_stats['total']:,}")
print(f"With narrative:         {narrative_stats['with_narrative']:,} ({narrative_stats['with_narrative_pct']:.1%})")
print(f"Without narrative:      {narrative_stats['without_narrative']:,}")

fig, ax = plt.subplots(figsize=(6, 4))
pd.Series(
    {
        "With Narrative": narrative_stats["with_narrative"],
        "Without Narrative": narrative_stats["without_narrative"],
    }
).plot(kind="bar", ax=ax, color=["steelblue", "salmon"], edgecolor="white")
ax.set_title("Complaints With vs Without Narratives (Full Dataset)")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/narrative_availability.png", dpi=120)
plt.show()

In [ ]:
raw_products = raw_product_distribution(RAW_PATH)
print("Top 10 raw product categories:")
print(raw_products.head(10))

fig, ax = plt.subplots(figsize=(10, 5))
raw_products.head(10).plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Top 10 Complaint Products (Raw, Before Filtering)")
ax.set_xlabel("Product")
ax.set_ylabel("Count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/raw_product_distribution.png", dpi=120)
plt.show()

## 2. Filter, clean, and save

Keep only Credit Card, Personal Loan, Savings Account, and Money Transfer complaints
with non-empty narratives. Clean text and drop very short entries.

In [ ]:
df = run_pipeline(
    raw_path=RAW_PATH,
    filtered_path=FILTERED_PATH,
    processed_path=PROCESSED_PATH,
)

print(f"Filtered dataset shape: {df.shape}")
print(df["product_category"].value_counts())
print(f"\nSaved to: {FILTERED_PATH}")
print(f"Saved to: {PROCESSED_PATH}")

## 3. EDA on the cleaned dataset

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df["product_category"].value_counts().plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Filtered Complaints by Product Category")
ax.set_xlabel("Product Category")
ax.set_ylabel("Number of Complaints")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/product_distribution.png", dpi=120)
plt.show()

In [ ]:
wc_stats = word_count_summary(df)
print("Word count summary (cleaned narratives):")
for key, value in wc_stats.items():
    print(f"  {key}: {value:.1f}" if isinstance(value, float) else f"  {key}: {value}")

fig, ax = plt.subplots(figsize=(10, 5))
df["word_count"].clip(upper=500).hist(bins=50, ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Consumer Narrative Word Count (clipped at 500)")
ax.set_xlabel("Word count")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/narrative_wordcount.png", dpi=120)
plt.show()

In [ ]:
df.head(3)[["complaint_id", "product_category", "issue", "word_count", "clean_narrative"]]